# Stage B only: full-sheet detector

Stage A is not trained. The classifier is `src/Object Detection/Component Training/models/custom_best.keras` from the GitHub clone.

Attach one dataset: `anayedeshan/sld-full-sheets`. GPU and Internet on.

From this PC: `python src/kaggle_runner/run.py sld-stage-b`


In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import Image as IPyImage, display


In [ ]:
REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = Path("/kaggle/working/Single-Line-Diagram-and-Energy-Mapping")
COMP_DIR = REPO / "src" / "Object Detection" / "Component Training"
SLD_DIR = REPO / "src" / "Object Detection" / "SLD Training"
CLASSIFIER = COMP_DIR / "models" / "custom_best.keras"
SAVE_DIR = Path("/kaggle/working/sld_pipeline")
INPUT = Path("/kaggle/input")
SLD_EPOCHS = 60


In [ ]:
def sh(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    return (r.stdout + r.stderr).strip()

def mounted(slug):
    name = slug.split("/")[-1]
    for c in (INPUT / "datasets" / slug, INPUT / name):
        if c.exists():
            return c
    return None

def find_tree(marker, under=None):
    base = under or INPUT
    if not base.exists():
        return None
    for hit in base.rglob(marker.split("/")[-1]):
        root = hit
        for _ in marker.split("/"):
            root = root.parent
        if (root / marker).exists():
            return root
    return None

def link(src, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink() or dest.exists():
        dest.unlink() if dest.is_symlink() else shutil.rmtree(dest)
    os.symlink(src, dest, target_is_directory=True)

def keep(src, saved_sub):
    if src.exists():
        shutil.copytree(src, SAVE_DIR / saved_sub, dirs_exist_ok=True)


In [ ]:
if (REPO / ".git").is_dir():
    sh("git", "pull", "--ff-only", cwd=REPO)
else:
    sh("git", "clone", "-q", REPO_URL, str(REPO))
assert CLASSIFIER.is_file(), f"classifier not in the GitHub clone: {CLASSIFIER}"
print("classifier", CLASSIFIER, CLASSIFIER.stat().st_size)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-image", "networkx"], check=True)


## Sheets and the committed classifier


In [ ]:
os.chdir(SLD_DIR)
sys.path.insert(0, str(SLD_DIR))
for name in ("config", "data", "model"):
    sys.modules.pop(name, None)
import config
from sld_data import load_classes, sheet_ids, image_path, load_graph
from graph_eval import score_sheet, summarise
from train_sld import train
from predict_graph import load_detector, predict_sheet
from reclassify import load_classifier, reclassify
from viz import show_graph

sheets_ds = mounted("anayedeshan/sld-full-sheets")
SHEETS = None
if sheets_ds:
    for c in (sheets_ds / "sld-sheets", sheets_ds):
        if (c / "data.yaml").is_file() and (c / "graphs").is_dir():
            SHEETS = c
            break
    if SHEETS is None:
        hit = find_tree("data.yaml", sheets_ds)
        if hit and (hit / "graphs").is_dir():
            SHEETS = hit
assert SHEETS, "sld-full-sheets not attached"
link(SHEETS, config.SHEETS_DIR)
classes = load_classes()
classifier = load_classifier(CLASSIFIER)
print(SHEETS)
print(len(classes), "detector classes |", len(classifier[1]), "classifier classes")
{s: len(sheet_ids(s)) for s in ("train", "val", "test")}


## Training samples

Three training sheets with the true symbol name on each box, before the detector is trained.

In [ ]:
from viz import show_labels

for sid in sheet_ids("train")[:3]:
    fig = show_labels(Image.open(image_path(sid, "train")), load_graph(sid), classes, f"train {sid}")
    display(fig)
    plt.close(fig)

## Train the sheet detector


In [ ]:
print(
    "aug: rotate±180 flip 0.5/0.5 scale 0.5 translate 0.2 shear 5 "
    "perspective 0.001 mosaic mixup 0.15 copy-paste 0.1 hsv"
)
WEIGHTS = train(
    epochs=SLD_EPOCHS,
    project=SAVE_DIR / "runs",
    degrees=180, fliplr=0.5, flipud=0.5, scale=0.5,
    translate=0.2, shear=5.0, perspective=0.001,
    mixup=0.15, copy_paste=0.1,
)
WEIGHTS


In [ ]:
from ultralytics import YOLO
from train_sld import resolve_data_yaml
data_yaml = resolve_data_yaml(config.DATA_YAML, SAVE_DIR / "data.yaml")
m = YOLO(str(WEIGHTS)).val(data=str(data_yaml), split="test", imgsz=config.IMG_SIZE, plots=False, verbose=False)
{"mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3)}


## Trace wires and re-label symbols with the GitHub classifier


In [ ]:
from tqdm.auto import tqdm

detector = load_detector(WEIGHTS)
test_ids = sheet_ids("test")

def recover(sid, use_classifier=True):
    img = Image.open(image_path(sid, "test"))
    result, graph = predict_sheet(detector, img, classes, sid)
    if use_classifier:
        reclassify(img, result["symbols"], classifier)
    return img, result, graph

rows_det, rows_clf = [], []
for s in tqdm(test_ids, desc="test sheets"):
    rows_det.append(score_sheet(load_graph(s), recover(s, use_classifier=False)[1]))
    rows_clf.append(score_sheet(load_graph(s), recover(s)[1]))
{"detector only": summarise(rows_det), "detector + classifier": summarise(rows_clf)}


In [ ]:
from viz import show_identified

for sid in test_ids[:4]:
    img, result, _ = recover(sid)
    fig, counts = show_identified(img, load_graph(sid), result["symbols"], classes, sid)
    print(f"{sid}  {counts['correct']}/{counts['gt']} correct, {counts['wrong']} wrong, {counts['missed']} missed")
    display(fig)
    plt.close(fig)


## Save detector weights and scores


In [ ]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
(config.OUTPUT_DIR / "graph_pred_test.json").write_text(json.dumps({
    "detector": summarise(rows_det),
    "detector+classifier": summarise(rows_clf),
}, indent=1))
keep(config.MODEL_DIR, "models/sld")
keep(config.OUTPUT_DIR, "outputs/sld")
(SAVE_DIR / "models" / "component").mkdir(parents=True, exist_ok=True)
shutil.copy2(CLASSIFIER, SAVE_DIR / "models" / "component" / "custom_best.keras")
bundle = Path("/kaggle/working/sld_stage_b.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in SAVE_DIR.rglob("*"):
        if path.is_file() and path.suffix != ".jpg":
            zf.write(path, path.relative_to(SAVE_DIR.parent))
print(bundle, bundle.stat().st_size)
